# Testbench notebook work in progress.

Currently it will **crash the kernel** after second run, when started in the test enviroment docker

Code in here is the same as other python scripts in this python folder. Until the kernel crashing behavior is fixed, no new code will be writen in the form of a jupyter notebook

In [ ]:
import math
import os
import sys
import random
sys.setdlopenflags(os.RTLD_NOW | os.RTLD_GLOBAL)
import oaipylib as oai
import numpy as np


def bpsk(inp, array_size):
    np_input = np.array(inp, dtype=np.uint32)
    # Create an array of shape (num_integers, 32)
    # Each row contains the bits of the corresponding integer
    bits = ((np_input[:, None] >> np.arange(32, dtype=np.uint32)) & 1).flatten()[:array_size]
    return (1 - 2 * bits.astype(np.int8)) / np.sqrt(2.0)

def awgn(samples, SNRdB):
    SNR_lin = 10**(SNRdB / 10.0)
    noise_variance = (1 / (2.0 * SNR_lin))
    sigma = np.sqrt(noise_variance)
    # Use a fixed seed for reproducible results
    random.seed(0)
    out = []
    for x in samples:
        noisy = x + random.gauss(0.0, sigma)
        q15 = int(round(noisy * 8.0))
        if q15 > 127:
            q15 = 127
        elif q15 < -128:
            q15 = -128
        out.append(q15)
    return np.array(out, dtype=np.int16)

# --- Test Script ---

oai.init()

# 1. Define input data
encoder_input = np.array([0x12345678], dtype='Q')

# 2. Call the OAI Polar Encoder
encoded_output = oai.nr_polar_encoder(encoder_input, 0, 0, 0, 32, 0)
encoded_len = 864

# 3. Modulate using BPSK
bpsk_symbols = bpsk(encoded_output, encoded_len)

# 4. Add AWGN and Quantize
SNRdB = 0
decoder_input = awgn(bpsk_symbols, SNRdB)

# 5. Call the OAI Polar Decoder
decoder_output = oai.nr_polar_decoder(decoder_input, 0, 0, encoded_len, 0)

# 6. Print result and shutdown
print(f"Original input:  {hex(encoder_input[0])}")
print(f"Decoded output:  {hex(decoder_output[0])}")
if encoder_input[0] == decoder_output[0]:
    print("SUCCESS: Decoded output matches original input.")
else:
    print("FAILURE: Decoded output does not match original input.")

oai.shutdown()


Pre-allocating padded host memory for the CPU channel pipeline...
Original input:  0x12345678
Decoded output:  0x12345678
SUCCESS: Decoded output matches original input.
[NR_MAC] TDD period index = 6, based on the sum of dl_UL_TransmissionPeriodicity from Pattern1 (5.000000 ms) and Pattern2 (0.000000 ms): Total = 5.000000 ms
[NR_MAC] Set TDD configuration period to: 8 DL slots, 3 UL slots, 10 slots per period (NR_TDD_UL_DL_Pattern is 7 DL slots, 2 UL slots, 6 DL symbols, 4 UL symbols)
[NR_MAC] Configured 1 TDD patterns (total slots: pattern1 = 10, pattern2 = 0)
[UTIL]   threadCreate() for MAC_STATS: creating thread (no affinity, default priority)
[NR_MAC] Set TX antenna number to 1, Set RX antenna number to 1 (num ssb 8: ff000000,0)
[NR_MAC] TDD period index = 6, based on the sum of dl_UL_TransmissionPeriodicity from Pattern1 (5.000000 ms) and Pattern2 (0.000000 ms): Total = 5.000000 ms
[NR_MAC] Set TDD configuration period to: 8 DL slots, 3 UL slots, 10 slots per period (NR_TDD_UL_DL_

[LOG] init aborted, configuration couldn't be performed


[NR_PHY] Set TDD Period Configuration: 2 periods per frame, 20 slots to be configured (8 DL, 3 UL)
[NR_PHY] TDD period configuration: slot 0 is DOWNLINK
[NR_PHY] TDD period configuration: slot 1 is DOWNLINK
[NR_PHY] TDD period configuration: slot 2 is DOWNLINK
[NR_PHY] TDD period configuration: slot 3 is DOWNLINK
[NR_PHY] TDD period configuration: slot 4 is DOWNLINK
[NR_PHY] TDD period configuration: slot 5 is DOWNLINK
[NR_PHY] TDD period configuration: slot 6 is DOWNLINK
[NR_PHY] TDD period configuration: slot 7 is FLEXIBLE: DDDDDDFFFFUUUU
[NR_PHY] TDD period configuration: slot 8 is UPLINK
[NR_PHY] TDD period configuration: slot 9 is UPLINK
[PHY]    DL frequency 3619080000 Hz, UL frequency 3619080000 Hz: uldl offset 0 Hz
[PHY]    Initializing frame parms for mu 1, N_RB 106, Ncp 0
[PHY]    Init: N_RB_DL 106, first_carrier_offset 1412, nb_prefix_samples 144,nb_prefix_samples0 176, ofdm_symbol_size 2048
[NR_MAC] TDA index 0: start 0 length 12 k2 6
[NR_MAC] TDA index 1: start 10 length 3